In [ ]:
import os, glob

base_path = "."  # match your existing base_path

# Remove every ACLiC-generated artifact for both classes
patterns = [
    "PhysdEdx_cpp*", "Hypfit_cpp*",
    "PhysdEdx_ACLiC_dict*", "Hypfit_ACLiC_dict*",
    "*.pcm", "*.d",
]
for pat in patterns:
    for f in glob.glob(os.path.join(base_path, pat)):
        print("removing", f)
        os.remove(f)

In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp

import os
import sys
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

print("Loading")
import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")


from analysis_village.cc1pi.var_configs import *

from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks.CutMasks import *
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from cols_to_keep import *

from analysis_village.cc1pi.TLExtensionMethod.GaussianFactorFittingUtils import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

In [ ]:
# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

import os
import ROOT
import subprocess
from ROOT import std

# 1. Setup Paths
base_path = "/home/lpelegri/cafpyana/analysis_village/cc1pi/TLExtensionMethod"
# Let's get the ROOT lib directory dynamically to ensure it's correct
try:
    root_lib_dir = subprocess.check_output(['root-config', '--libdir'], text=True).strip()
except:
    root_lib_dir = "/cvmfs/larsoft.opensciencegrid.org/products/root/v6_28_12/Linux64bit+3.10-2.17-e26-p3915-prof/lib"

# 2. Update Environment
os.environ['LD_LIBRARY_PATH'] = f"{root_lib_dir}:{os.environ.get('LD_LIBRARY_PATH', '')}"
ROOT.gSystem.AddDynamicPath(root_lib_dir)

# 3. Load the "Big" ROOT dependency blocks
# Loading these covers almost all physics class requirements (Hist, Geom, Graf, etc.)
root_libs = ["libRint","libROOTGpadv7","libMathMore","libCore", "libRIO", "libNet", "libHist", "libGraf", "libGraf3d", "libGpad", "libTree", "libMathCore", "libThread", "libMatrix", "libGeom","libROOTHist"]
for lib in root_libs:
    ROOT.gSystem.Load(lib)

def load_custom_class(class_name):
    source_file = os.path.join(base_path, f"{class_name}.cpp")
    header_file = os.path.join(base_path, f"{class_name}.h")
    so_file = os.path.join(base_path, f"{class_name}_cpp.so")

    status = ROOT.gSystem.CompileMacro(source_file, "kOf")
    if status >= 0:
        if ROOT.gSystem.Load(so_file) < 0:
            print(f"⚠️  Compiled {class_name} but failed to load {so_file}")
            return False
        ROOT.gInterpreter.Declare(f'#include "{header_file}"')  # keep this!
        print(f"📦 Successfully compiled and linked: {class_name}")
        return True
    else:
        print(f"❌ Failed to compile: {class_name}")
        return False

if load_custom_class("PhysdEdx"):
    if load_custom_class("Hypfit"):
        try:
            ROOT.gInterpreter.ProcessLine(f'#include "{os.path.join(base_path, "Hypfit.h")}"')
            h_fit = ROOT.Hypfit()
            print("🚀 Success! Hypfit object initialized and linked.")
        except Exception as e:
            print(f"❌ Error during instantiation: {e}")
            ROOT.gInterpreter.ProcessLine("Hypfit* h_fit_ptr = new Hypfit();")
            h_fit = ROOT.h_fit_ptr
            print("🚀 Success! Hypfit object initialized via Global Pointer fallback.")

In [ ]:
# ======================================================================
# Configuration
# ======================================================================
PARTICLE = "pion"              # target particle: "pion" or "muon"
PION_SHIFT_FROM_MUONS = True   # extend the pion shift with muon slices at equal betagamma

PDGS = {"muon": 13, "pion": 211}
PDG = PDGS[PARTICLE]
PDG_MASS = {13: 105.6583755, 211: 139.57039, 2212: 938.27208816}   # MeV

MC_FILES = {
    "muon": "/exp/sbnd/data/users/lpelegri/TLECafpyanaData_gen1/TLE_1e20_muon_update_calo.df",
    "pion": "/exp/sbnd/data/users/lpelegri/TLECafpyanaData_gen1/TLE_1e20_pion_stopping_update_calo.df",
}
N_FILES = 30                                                  # files loaded per sample
KEYS2LOAD = ["pfp", "hdr", "histpotdf", "hit0", "hit1", "hit2"]
SAMPLES = ("MC",)

# Slice fits
SLICE_RR_MAX = {"pion": 80.0, "muon": 250.0}   # upper residual range of the slices [cm]
SLICE_PITCH = 0.3                              # pitch of the theoretical PDF [cm]
SLICE_SIGMA_ERR_RANGE = (1e-4, 0.2)            # sigma_G error of the slices used for sigma_G [MeV/cm]

# Muon correction of the pion shift (pion residual range [cm])
PION_OVERLAP = (40.0, 80.0)    # window where the pion - muon offset is measured
PION_RR_JOIN = 80.0            # pion slices up to here, muon slices + offset beyond

PARAM_TPC = -1                 # TPC of the parameters for PhysdEdx.cpp (-1: both TPCs)

FIG_DIR = "/exp/sbnd/data/users/lpelegri/TLEGraphs/PDFExtractionMuonExtension"

# Particles to load and fit: the target particle and, for the muon correction, the muons
PLOT_PARTICLES = [PARTICLE] + (["muon"] if PARTICLE == "pion" and PION_SHIFT_FROM_MUONS else [])

In [ ]:
# ======================================================================
# Plot style, output folder and helpers shared by all the plots
# ======================================================================
PLANE_COLORS = {0: "tab:red", 1: "tab:blue", 2: "tab:green"}
TPC_COLORS = {0: "tab:red", 1: "tab:blue", -1: "tab:green"}
TPC_STYLES = {
    0: {"linestyle": "--", "marker": "o", "label_prefix": "x < 0 (TPC 0)"},
    1: {"linestyle": ":", "marker": "s", "label_prefix": "x > 0 (TPC 1)"},
    -1: {"linestyle": "-", "marker": "^", "label_prefix": "Combined"},
}
SAMPLE_COLORS = {"MC": "#d62728", "Data": "#1f77b4"}   # pion points in the correction steps
MUON_COLOR = "#7f7f7f"                                   # muon points in the correction steps
SHIFT_LABEL = r"Shift $\Delta$MPV [MeV/cm]"
SIGMA_LABEL = r"$\sigma_G$ [MeV/cm]"


def save_figure(fig, name):
    """Save fig as FIG_DIR/name, creating FIG_DIR if needed."""
    os.makedirs(FIG_DIR, exist_ok=True)
    path = os.path.join(FIG_DIR, name)
    fig.savefig(path, bbox_inches="tight")
    print(f"Saved {path}")


def sigma_slices(particle, sample, plane, tpc=PARAM_TPC):
    """Slices used for the sigma_G fit (sigma_ok slices when flagged), sorted by residual
    range; empty if there are none."""
    res = slice_results[(particle, sample)].get((plane, tpc), pd.DataFrame())
    if not len(res):
        return res
    if "sigma_ok" in res:
        res = res[res["sigma_ok"]]
    return res.dropna(subset=["rr_center", "mpv_theory", "gsigma"]).sort_values("rr_center")

In [ ]:
# ======================================================================
# Loading of one MC sample, and checks of its structure and particle content
# ======================================================================
TRACK_LEVELS = ["__ntuple", "entry", "rec.slc..index", "rec.slc.reco.pfp..index"]
P_TYPE_COL = ("pfp", "trk", "truth", "p", "p_type", "")
WEIGHT_COL = ("slc", "wgt", "", "", "", "")     # None for unweighted counts


def load_mc_sample(particle):
    """Data frames (KEYS2LOAD) of the MC sample of one particle."""
    path = MC_FILES[particle]
    print(f"Loading {path}")
    return load_df(path, KEYS2LOAD, N_FILES)


def _track_set(df):
    """Unique TRACK_LEVELS index combinations (tracks) of df."""
    return set(df.index.to_frame()[TRACK_LEVELS].itertuples(index=False, name=None))


def check_tracks(dfs):
    """Hit rows and tracks in hit0 + hit1 + hit2, tracks in pfp, and tracks in both."""
    hit_keys = [k for k in ("hit0", "hit1", "hit2") if not dfs[k].empty]
    hit_tracks = set().union(*(_track_set(dfs[k]) for k in hit_keys))
    pfp_tracks = _track_set(dfs["pfp"])
    print(f"hit0 + hit1 + hit2: {sum(len(dfs[k]) for k in hit_keys)} hit rows, "
          f"{len(hit_tracks)} unique tracks")
    print(f"pfp: {len(dfs['pfp'])} rows, {len(pfp_tracks)} unique tracks")
    print(f"Tracks in both pfp and hits: {len(hit_tracks & pfp_tracks)}")


def p_type_summary(pfp_df):
    """Raw and weighted counts of the true particle type of the pfp tracks."""
    df = pfp_df[pfp_df[P_TYPE_COL].notna()]
    if WEIGHT_COL is not None and WEIGHT_COL in df.columns:
        weights = df[WEIGHT_COL].fillna(1.0)
    else:
        weights = pd.Series(1.0, index=df.index)
    summary = (pd.DataFrame({"p_type": df[P_TYPE_COL], "weight": weights})
               .groupby("p_type")
               .agg(Counts=("weight", "count"), Weighted_Yield=("weight", "sum"))
               .reset_index())
    total_counts, total_weighted = summary["Counts"].sum(), summary["Weighted_Yield"].sum()
    summary["Raw_%"] = 100.0 * summary["Counts"] / total_counts
    summary["Weighted_%"] = 100.0 * summary["Weighted_Yield"] / total_weighted
    summary = summary.sort_values("Weighted_Yield", ascending=False)

    print("=" * 60)
    print(f"{'p_type':<15} | {'Counts':<8} | {'Raw %':<8} | {'Weighted':<10} | {'Weighted %':<10}")
    print("-" * 60)
    for _, row in summary.iterrows():
        print(f"{str(row['p_type']):<15} | {int(row['Counts']):<8d} | {row['Raw_%']:<7.2f}% | "
              f"{row['Weighted_Yield']:<10.1f} | {row['Weighted_%']:<9.2f}%")
    print("-" * 60)
    print(f"{'Total':<15} | {int(total_counts):<8d} | {100.0:<7.2f}% | "
          f"{total_weighted:<10.1f} | {100.0:<9.2f}%")
    print("=" * 60)

In [ ]:
# ======================================================================
# dE/dx slice fits: theoretical PDF (x) Gaussian(sigma_G), shifted by Delta
# ======================================================================
def build_theoretical_pdf(pdg, rr_center, pitch=SLICE_PITCH):
    """TF1 of the theoretical dE/dx PDF at residual range rr_center [cm]
    (same construction as Hypfit::get_conv_function_map)."""
    phys = h_fit.map_PhysdEdx[pdg]
    mass = PDG_MASS[abs(pdg)]
    ke = phys.KEFromRangeSpline(float(rr_center))
    gamma = ke / mass + 1.0
    beta2 = 1.0 - 1.0 / gamma**2
    xi = phys.Landau_xi(ke, pitch)
    kappa = xi / phys.Get_Wmax(ke)
    pdf = ROOT.TF1("", ROOT.PhysdEdx.dEdx_PDF_function, -10.0, 20.0, 5)
    pdf.SetParameters(kappa, beta2, xi, phys.meandEdx(ke), pitch)
    return pdf


def theory_mpv(pdf):
    """Peak of the theoretical PDF, with the same scan as the maps (Hypfit::robust_max_x)."""
    try:
        return float(h_fit.robust_max_x(pdf, 0.0, 10.0, 10000))
    except Exception:
        xs = np.linspace(0.0, 10.0, 10000)
        return float(xs[int(np.argmax([pdf.Eval(x) for x in xs]))])


_THEORY_CACHE = {}


def theory_at(pdg, rr_center, pitch=SLICE_PITCH, x_min=-5.0, x_max=25.0, n_points=6001):
    """(MPV, x grid, PDF values on the grid, grid step), cached per residual range."""
    key = (pdg, round(float(rr_center), 4), pitch)
    if key not in _THEORY_CACHE:
        pdf = build_theoretical_pdf(pdg, rr_center, pitch)
        x_grid = np.linspace(x_min, x_max, n_points)
        pdf_vals = np.array([pdf.Eval(x) for x in x_grid])
        _THEORY_CACHE[key] = (theory_mpv(pdf), x_grid, pdf_vals, x_grid[1] - x_grid[0])
    return _THEORY_CACHE[key]


def gaussian_kernel(x, sigma):
    """Zero-mean Gaussian sampled on x (normalized by the caller)."""
    if sigma <= 0:
        kernel = np.zeros_like(x)
        kernel[np.argmin(np.abs(x))] = 1.0
        return kernel
    return np.exp(-0.5 * (x / sigma) ** 2)


def convolved_theoretical_pdf(x_query, sigma, shift, amplitude, x_grid, pdf_vals, dx):
    """amplitude * (PDF (x) unit-area Gaussian(sigma))(x - shift)."""
    x_centered = x_grid - x_grid[len(x_grid) // 2]
    kernel = gaussian_kernel(x_centered, sigma)
    kernel = kernel / (kernel.sum() * dx)
    conv = fftconvolve(pdf_vals, kernel, mode="same") * dx
    return amplitude * np.interp(x_query - shift, x_grid, conv)


def fit_theoretical_conv_slice(bin_centers, bin_counts, bin_errs, x_grid, pdf_vals, dx,
                               mpv_theory, fit_range=None, sigma0=0.2,
                               sigma_bounds=(1e-4, 5.0), shift_bounds=(-3.0, 3.0)):
    """Fits (sigma, shift, amplitude) in [0.6, 1.4] x MPV; also returns chi2, ndf and
    the sigma-shift correlation. None if the fit fails or has fewer than 5 bins."""
    if fit_range is None:
        fit_range = (0.6 * mpv_theory, 1.4 * mpv_theory)
    mask = (bin_centers >= fit_range[0]) & (bin_centers <= fit_range[1]) & (bin_counts > 0)
    if mask.sum() < 5:
        return None
    xs, ys = bin_centers[mask], bin_counts[mask]
    yerr = np.where(bin_errs[mask] > 0, bin_errs[mask], 1.0)
    amp0 = ys.max() / (pdf_vals.max() if pdf_vals.max() > 0 else 1.0)

    def model(x, sigma, shift, amplitude):
        return convolved_theoretical_pdf(x, sigma, shift, amplitude, x_grid, pdf_vals, dx)

    try:
        popt, pcov = curve_fit(
            model, xs, ys, p0=[sigma0, 0.0, amp0], sigma=yerr, absolute_sigma=True,
            bounds=([sigma_bounds[0], shift_bounds[0], 0.0],
                    [sigma_bounds[1], shift_bounds[1], np.inf]),
            maxfev=20000,
        )
    except Exception:
        return None

    perr = np.sqrt(np.diag(pcov))
    chi2 = float(np.sum(((ys - model(xs, *popt)) / yerr) ** 2))
    denom = np.sqrt(pcov[0, 0] * pcov[1, 1])
    corr = pcov[0, 1] / denom if np.isfinite(denom) and denom > 0 else np.nan
    return dict(sigma=popt[0], sigma_err=perr[0], shift=popt[1], shift_err=perr[1],
                amplitude=popt[2], amplitude_err=perr[2], chi2=chi2,
                ndf=len(xs) - len(popt), corr_sigma_shift=corr, fit_range=fit_range)


def fit_rr_slices_theoretical(df, plane, tpc, pdg, rr_min=3.0, rr_max=40.0, rr_bin_width=1.0,
                              hist_nbins=150, hist_range=(0.0, 20.0), min_entries=30):
    """Slice fits of one (plane, tpc) hit DataFrame (columns rr, dedx). Keeps every
    converged fit with a finite shift error; the sigma_G quality cut is applied as a
    flag in run_slice_fits."""
    edges = np.arange(rr_min, rr_max + rr_bin_width, rr_bin_width)
    n_slices = len(edges) - 1
    rr = df["rr"].to_numpy(dtype=float)
    dedx = df["dedx"].to_numpy(dtype=float)
    ok = np.isfinite(rr) & np.isfinite(dedx) & (rr >= edges[0]) & (rr < edges[-1])

    # Sort the hits by residual-range slice once, instead of selecting each slice
    ibin = np.floor((rr[ok] - edges[0]) / rr_bin_width).astype(int)
    order = np.argsort(ibin, kind="stable")
    ibin, dedx_sorted = ibin[order], dedx[ok][order]
    bounds = np.searchsorted(ibin, np.arange(n_slices + 1))

    hist_edges = np.linspace(hist_range[0], hist_range[1], hist_nbins + 1)
    bin_centers = 0.5 * (hist_edges[:-1] + hist_edges[1:])
    rows = []
    for i in range(n_slices):
        vals = dedx_sorted[bounds[i]:bounds[i + 1]]
        if len(vals) < min_entries:
            continue
        rr_center = 0.5 * (edges[i] + edges[i + 1])
        mpv_theory, x_grid, pdf_vals, dx = theory_at(pdg, rr_center)
        counts = np.histogram(vals, bins=hist_edges)[0].astype(float)
        errs = np.where(counts > 0, np.sqrt(counts), 1.0)
        fit = fit_theoretical_conv_slice(bin_centers, counts, errs, x_grid, pdf_vals, dx, mpv_theory)
        if fit is None or not np.isfinite(fit["shift_err"]):
            continue
        rows.append(dict(
            plane=plane, tpc=tpc, rr_center=rr_center, n_hits=len(vals),
            mpv_x=mpv_theory, mpv_theory=mpv_theory,
            gsigma=fit["sigma"], gsigma_err=fit["sigma_err"],
            shift=fit["shift"], shift_err=fit["shift_err"],
            chi2=fit["chi2"], ndf=fit["ndf"], corr_sigma_shift=fit["corr_sigma_shift"],
            fit_range_lo=fit["fit_range"][0], fit_range_hi=fit["fit_range"][1],
        ))
    return pd.DataFrame(rows)


def run_slice_fits(hit_dfs, pdg, rr_max):
    """Slice fits for planes 0-2 and TPC 0, 1 and both combined (tpc = -1).
    Every converged fit is kept; sigma_ok flags the slices whose sigma_G error lies
    in SLICE_SIGMA_ERR_RANGE. Returns {(plane, tpc): DataFrame}."""
    lo, hi = SLICE_SIGMA_ERR_RANGE
    results = {}
    for plane, df in enumerate(hit_dfs):
        for tpc in (0, 1, -1):
            sub = df if tpc == -1 else df[df["tpc"] == tpc]
            res = fit_rr_slices_theoretical(sub, plane, tpc, pdg, rr_max=rr_max)
            if len(res):
                e = res["gsigma_err"]
                res["sigma_ok"] = np.isfinite(e) & (e >= lo) & (e <= hi)
            n_ok = int(res["sigma_ok"].sum()) if len(res) else 0
            results[(plane, tpc)] = res
            print(f"  plane {plane}, TPC {'both' if tpc == -1 else tpc}: "
                  f"{len(res)} slices, {n_ok} with usable sigma_G")
    return results

In [ ]:
# ======================================================================
# Load each MC sample, check it, and fit its dE/dx slices
# (one sample in memory at a time; the hits are freed after their slice fits)
# ======================================================================
slice_results = {}    # (particle, sample) -> {(plane, tpc): slice fits}
for particle in PLOT_PARTICLES:
    print(f"\n{'#' * 20} {particle} MC {'#' * 20}")
    dfs = load_mc_sample(particle)
    check_tracks(dfs)
    p_type_summary(dfs["pfp"])
    print(f"Slice fits up to {SLICE_RR_MAX[particle]:g} cm")
    slice_results[(particle, "MC")] = run_slice_fits([dfs["hit0"], dfs["hit1"], dfs["hit2"]],
                                                     PDGS[particle], SLICE_RR_MAX[particle])
    del dfs
    gc.collect()

In [ ]:
# ======================================================================
# Fit models and fit functions: sigma_G vs MPV, shift vs residual range, and the
# pion - muon offset (muon shift from a local fit around PION_OVERLAP)
# ======================================================================
def power_law(x, a, b, c):
    return a + b * np.power(x, c)


def exp_decay_plateau(rr, a, b, c):
    """a + b * exp(-rr / c)"""
    return a + b * np.exp(-rr / c)


def exp_decay_linear(rr, a, b, c, d):
    """a + b * exp(-rr / c) + d * rr"""
    return a + b * np.exp(-rr / c) + d * rr


def power_law_linear(rr, a, b, p, d):
    """a + b * rr^p + d * rr"""
    return a + b * np.power(rr, p) + d * rr


# Shift models: name -> (function, parameter names)
SHIFT_MODELS = {
    "exp_plateau": (exp_decay_plateau, ("a", "b", "c")),
    "exp_linear": (exp_decay_linear, ("a", "b", "c", "d")),
    "power_linear": (power_law_linear, ("a", "b", "p", "d")),
}
SHIFT_MODEL = "power_linear"   # shift model of every shift fit, a key of SHIFT_MODELS


def eval_shift_fit(x, popt, model, rr_floor=None):
    """Shift model `model` at residual range x. With rr_floor, the model is evaluated
    at max(x, rr_floor), so it stays constant below rr_floor."""
    x = np.asarray(x, dtype=float)
    if rr_floor is not None:
        x = np.maximum(x, rr_floor)
    return SHIFT_MODELS[model][0](x, *popt)


def shift_fit_params(popt, perr, model):
    """{"model": model, name: value, name_err: error} for the shift-fit parameters."""
    out = {"model": model}
    for name, val, err in zip(SHIFT_MODELS[model][1], popt, perr):
        out[name], out[name + "_err"] = float(val), float(err)
    return out


def fit_sigma_vs_mpv(res, p0=(0.05, 0.1, 2.0),
                     bounds=((0.0, 0.0, -20.0), (2.0, 100.0, 20.0))):
    """sigma_G = a + b * MPV^c, MPV = theoretical MPV of the slice. Uses only the
    slices flagged sigma_ok by run_slice_fits. Returns (popt, perr)."""
    v = res.dropna(subset=["mpv_theory", "gsigma", "gsigma_err"])
    v = v[v["gsigma_err"] > 0]
    if "sigma_ok" in v:
        v = v[v["sigma_ok"]]
    if len(v) < 4:
        return None, None
    try:
        popt, pcov = curve_fit(power_law, v["mpv_theory"].to_numpy(), v["gsigma"].to_numpy(),
                               p0=p0, sigma=v["gsigma_err"].to_numpy(), absolute_sigma=True,
                               bounds=bounds, maxfev=20000)
    except Exception as exc:
        print(f"  sigma fit failed: {exc}")
        return None, None
    return popt, np.sqrt(np.diag(pcov))


def fit_shift_vs_rr(points, model="exp_linear", c_max=50.0, p_bounds=(-4.0, -0.05),
                    verbose=False):
    """Shift vs residual range. model: "exp_linear" (a, b, c, d), "exp_plateau" (a, b, c)
    or "power_linear" (a, b, p, d). c_max bounds the decay length, so the exponential
    stays the fast component; p_bounds bounds the power-law exponent."""
    if model not in SHIFT_MODELS:
        raise ValueError(f"Unknown shift model: {model}")
    func, names = SHIFT_MODELS[model]
    v = points.dropna(subset=["rr_center", "shift", "shift_err"])
    v = v[v["shift_err"] > 0].sort_values("rr_center")
    if len(v) < len(names) + 2:
        return None, None
    x, y, yerr = (v[c].to_numpy() for c in ("rr_center", "shift", "shift_err"))

    c0, p_start = 5.0, -1.0
    if model in ("exp_linear", "power_linear"):
        tail = x >= x.min() + (x.max() - x.min()) / 3.0
        d0, a0 = np.polyfit(x[tail], y[tail], 1, w=1.0 / yerr[tail])
        excess0 = y[0] - (a0 + d0 * x[0])
    if model == "exp_linear":
        p0 = [a0, excess0 * np.exp(x[0] / c0), c0, d0]
        bounds = ([-np.inf, -np.inf, 0.1, -np.inf], [np.inf, np.inf, c_max, np.inf])
    elif model == "power_linear":
        p0 = [a0, excess0 * np.power(x[0], -p_start), p_start, d0]
        bounds = ([-np.inf, -np.inf, p_bounds[0], -np.inf],
                  [np.inf, np.inf, p_bounds[1], np.inf])
    else:
        p0 = [y[-1], y[0] - y[-1], c0]
        bounds = ([-np.inf, -np.inf, 0.1], [np.inf, np.inf, c_max])

    try:
        popt, pcov = curve_fit(func, x, y, p0=p0, sigma=yerr, absolute_sigma=True,
                               bounds=bounds, maxfev=20000)
    except Exception as exc:
        print(f"  shift fit failed: {exc}")
        return None, None
    if verbose:
        chi2 = np.sum(((y - func(x, *popt)) / yerr) ** 2)
        print(f"  shift fit ({model}): chi2/ndf = {chi2:.1f}/{len(x) - len(popt)}")
    return popt, np.sqrt(np.diag(pcov))


# ---- Muon shifts transferred to pions at equal betagamma ----
def rr_to_betagamma(pdg, rr):
    """betagamma at residual range rr [cm]."""
    phys = h_fit.map_PhysdEdx[pdg]
    mass = PDG_MASS[abs(pdg)]
    return np.array([phys.KEtoMomentum(phys.KEFromRangeSpline(float(r))) / mass for r in rr])


def betagamma_to_rr(pdg, bg):
    """Residual range [cm] at which particle pdg has the given betagamma."""
    phys = h_fit.map_PhysdEdx[pdg]
    mass = PDG_MASS[abs(pdg)]
    return np.array([phys.RangeFromKESpline(phys.MomentumtoKE(float(b) * mass)) for b in bg])


def muon_points_as_pion(mu_res):
    """Muon shift points with rr_center replaced by the pion residual range of equal betagamma."""
    out = mu_res[["rr_center", "shift", "shift_err"]].dropna().copy()
    out["rr_center"] = betagamma_to_rr(211, rr_to_betagamma(13, out["rr_center"].to_numpy()))
    return out.sort_values("rr_center").reset_index(drop=True)


MUON_LOCAL_FIT_MARGIN = 2.0   # pion residual range [cm] added on each side of PION_OVERLAP
                              # for the local muon fit


def muon_offset(pi_res, mu_res, overlap=PION_OVERLAP, margin=MUON_LOCAL_FIT_MARGIN, model=None):
    """Pion - muon shift offset at equal betagamma, with the muon shift from a local fit.
    The muon slices whose pion residual range of equal betagamma lies inside `overlap`
    widened by `margin` on each side are fitted in muon residual range with the shift model
    (fit_shift_vs_rr, as for the whole muon range). The fit is evaluated at the muon residual
    range of equal betagamma of each pion slice inside `overlap`, and the offset is the
    weighted mean pion - muon difference (weights 1 / pion error^2).
    Returns a dict with the offset and the quantities it is computed from."""
    model = model or SHIFT_MODEL
    lo, hi = overlap
    mu_own = mu_res[["rr_center", "shift", "shift_err"]].dropna()
    mu_own = mu_own[mu_own["shift_err"] > 0]
    rr_as_pion = betagamma_to_rr(211, rr_to_betagamma(13, mu_own["rr_center"].to_numpy()))
    local = mu_own[(rr_as_pion >= lo - margin) & (rr_as_pion <= hi + margin)]
    popt, _ = fit_shift_vs_rr(local, model=model)
    if popt is None:
        raise RuntimeError(f"Local muon fit failed ({len(local)} muon slices around PION_OVERLAP)")

    pi = pi_res[["rr_center", "shift", "shift_err"]].dropna()
    ov = pi[(pi["rr_center"] >= lo) & (pi["rr_center"] <= hi)]
    if len(ov) < 2:
        raise RuntimeError("Fewer than 2 pion slices in PION_OVERLAP")
    r_pi = ov["rr_center"].to_numpy()
    r_mu = betagamma_to_rr(13, rr_to_betagamma(211, r_pi))
    if r_mu.min() < local["rr_center"].min() or r_mu.max() > local["rr_center"].max():
        raise RuntimeError("The muon slices do not cover PION_OVERLAP: "
                           "check PION_OVERLAP and SLICE_RR_MAX")

    mu_fit = eval_shift_fit(r_mu, popt, model)
    pi_shift, pi_err = ov["shift"].to_numpy(), ov["shift_err"].to_numpy()
    diff = pi_shift - mu_fit
    w = 1.0 / pi_err ** 2
    return dict(offset=float(np.sum(w * diff) / np.sum(w)), r_pi=r_pi, r_mu=r_mu,
                pi_shift=pi_shift, pi_err=pi_err, mu_fit=mu_fit, diff=diff,
                mu_local=local, popt=popt, model=model)


def combined_shift_points(pi_res, mu_res, rr_join=PION_RR_JOIN, overlap=PION_OVERLAP):
    """Pion shift points up to rr_join, then muon points at equal betagamma beyond it,
    moved by the pion - muon offset of muon_offset (muon shift from a local fit).
    Returns (points, offset)."""
    pi = pi_res[["rr_center", "shift", "shift_err"]].dropna()
    mu = muon_points_as_pion(mu_res)
    offset = muon_offset(pi_res, mu_res, overlap)["offset"]
    mu_far = mu[mu["rr_center"] > rr_join].assign(shift=lambda d: d["shift"] + offset)
    points = pd.concat([pi[pi["rr_center"] <= rr_join], mu_far], ignore_index=True)
    return points.sort_values("rr_center").reset_index(drop=True), offset

In [ ]:
# ======================================================================
# Fits of sigma_G and of the shift for every (sample, plane, TPC), and the parameters
# at PARAM_TPC that are saved for PhysdEdx.cpp
# ======================================================================
TPC_LIST = (1, 0, -1)    # TPC 1 (X > 0), TPC 0 (X < 0), both TPCs combined


def tpc_shift_points(particle, sample, plane, tpc):
    """Shift points of one (sample, plane, TPC): the slices, or for pions with
    PION_SHIFT_FROM_MUONS the pion slices joined to the offset muon slices."""
    res = slice_results[(particle, sample)].get((plane, tpc), pd.DataFrame())
    if not len(res):
        return pd.DataFrame()
    if particle == "pion" and PION_SHIFT_FROM_MUONS:
        mu_res = slice_results[("muon", sample)].get((plane, tpc), pd.DataFrame())
        if not len(mu_res):
            print(f"[pion {sample}] plane {plane}, TPC {tpc}: no muon slice fits, shift skipped")
            return pd.DataFrame()
        try:
            points, _ = combined_shift_points(res, mu_res)
        except RuntimeError as exc:
            print(f"[pion {sample}] plane {plane}, TPC {tpc}: {exc}; shift skipped")
            return pd.DataFrame()
        return points
    return res[["rr_center", "shift", "shift_err"]].dropna()


def fit_per_tpc(particle):
    """sigma_G and shift fits of every (plane, TPC) with slice fits. Returns
    sample -> {"sigma": {(plane, tpc): (popt, perr)}, "shift": {(plane, tpc): (popt, perr)},
               "points": {(plane, tpc): shift points}, "rr_floor": {tpc: cm},
               "rr_ceil": {tpc: cm}, "shift_model": name}.
    rr_floor (rr_ceil) is the largest first (smallest last) shift point over the planes."""
    fits = {}
    for sample in SAMPLES:
        sigma, shift, points, rr_floor, rr_ceil = {}, {}, {}, {}, {}
        for tpc in TPC_LIST:
            rr_start, rr_end = [], []
            for plane in range(3):
                res = slice_results[(particle, sample)].get((plane, tpc), pd.DataFrame())
                if not len(res):
                    print(f"[{particle} {sample}] plane {plane}, TPC {tpc}: no slice fits")
                    continue
                sigma[(plane, tpc)] = fit_sigma_vs_mpv(res)
                pts = tpc_shift_points(particle, sample, plane, tpc)
                if not len(pts):
                    continue
                points[(plane, tpc)] = pts
                shift[(plane, tpc)] = fit_shift_vs_rr(pts, model=SHIFT_MODEL)
                rr_start.append(pts["rr_center"].min())
                rr_end.append(pts["rr_center"].max())
            if rr_start:
                rr_floor[tpc], rr_ceil[tpc] = float(np.max(rr_start)), float(np.min(rr_end))
        fits[sample] = dict(sigma=sigma, shift=shift, points=points, rr_floor=rr_floor,
                            rr_ceil=rr_ceil, shift_model=SHIFT_MODEL)
    return fits


def params_at_tpc(fits, tpc=PARAM_TPC):
    """Fits of one particle at one TPC, in the layout of the saved parameters:
    sample -> {"sigma": {plane: popt}, "shift": {plane: popt}, "shift_model": name,
               "rr_shift_min": cm, "rr_shift_max": cm}."""
    out = {}
    for sample, f in fits.items():
        missing = [plane for plane in range(3) if (plane, tpc) not in f["points"]]
        if missing:
            raise RuntimeError(f"No shift points for {sample}, TPC {tpc}, planes {missing}")
        out[sample] = dict(sigma={plane: f["sigma"][(plane, tpc)][0] for plane in range(3)},
                           shift={plane: f["shift"][(plane, tpc)][0] for plane in range(3)},
                           shift_model=f["shift_model"],
                           rr_shift_min=f["rr_floor"][tpc], rr_shift_max=f["rr_ceil"][tpc])
    return out


def points_at_tpc(fits, tpc=PARAM_TPC):
    """(sample, plane) -> shift points of one particle at one TPC."""
    return {(sample, plane): f["points"][(plane, tpc)]
            for sample, f in fits.items() for plane in range(3)}


def print_shift_fits(particle, fits, tpc=PARAM_TPC):
    """Number and range of the shift points, and the shift-fit parameters, at one TPC."""
    for sample, f in fits.items():
        for plane in range(3):
            pts = f["points"][(plane, tpc)]
            popt, perr = f["shift"][(plane, tpc)]
            print(f"[{particle} {sample}] plane {plane}: {len(pts)} shift points, "
                  f"{pts['rr_center'].min():.0f}-{pts['rr_center'].max():.0f} cm")
            if popt is None:
                print("    shift fit failed")
                continue
            pars = shift_fit_params(popt, perr, f["shift_model"])
            print("    " + ", ".join(f"{n} = {pars[n]:.4g} +- {pars[n + '_err']:.2g}"
                                     for n in SHIFT_MODELS[f["shift_model"]][1]))


tpc_fits = {particle: fit_per_tpc(particle) for particle in PLOT_PARTICLES}

# Saved parameters: fits at PARAM_TPC of the target particle and, when fitted, of the muons
fit_params = params_at_tpc(tpc_fits[PARTICLE])
shift_points = points_at_tpc(tpc_fits[PARTICLE])
mu_params, mu_points = {}, {}
if PARTICLE == "pion" and PION_SHIFT_FROM_MUONS:
    mu_params = params_at_tpc(tpc_fits["muon"])
    mu_points = points_at_tpc(tpc_fits["muon"])

for particle in PLOT_PARTICLES:
    print_shift_fits(particle, tpc_fits[particle])

In [ ]:
# ======================================================================
# Pion - muon offset of the shift, step by step: one figure per sample,
# one row per step, one column per plane
# ======================================================================
EB = dict(ms=4, elinewidth=0.7, capsize=0)


def offset_inputs(sample, plane, tpc=PARAM_TPC):
    """muon_offset of one sample and plane, the pion and muon slices, and the points of the
    pion shift fit."""
    pi_res = slice_results[("pion", sample)][(plane, tpc)]
    mu_res = slice_results[("muon", sample)][(plane, tpc)]
    o = muon_offset(pi_res, mu_res)
    o["points"] = tpc_fits["pion"][sample]["points"][(plane, tpc)]
    o["pi"] = pi_res[["rr_center", "shift", "shift_err"]].dropna()
    o["mu_own"] = mu_res[["rr_center", "shift", "shift_err"]].dropna()
    o["mu"] = muon_points_as_pion(mu_res)
    o["sample"], o["plane"], o["tpc"] = sample, plane, tpc
    return o


def _within(df, x_max):
    return df if x_max is None else df[df["rr_center"] <= x_max]


def _errorbar(ax, df, fmt, color, label, filled=True, alpha=0.85):
    ax.errorbar(df["rr_center"], df["shift"], yerr=df["shift_err"], fmt=fmt, color=color,
                mfc=color if filled else "none", alpha=alpha, label=label, **EB)


def _window_and_join(ax):
    ax.axvspan(*PION_OVERLAP, color="gray", alpha=0.15, label="Offset window")
    ax.axvline(PION_RR_JOIN, color="gray", ls="-.", lw=1, label="End of pion points")


def _text(ax, text):
    """Text box at the top right, with headroom added above the points."""
    y_lo, y_hi = ax.get_ylim()
    ax.set_ylim(y_lo, y_hi + 0.3 * (y_hi - y_lo))
    ax.text(0.97, 0.97, text, transform=ax.transAxes, fontsize=12, va="top", ha="right",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.85, edgecolor="none"))


# ---- One function per step, drawing one plane on one axis ----
def _step1(ax, o, x_max=None, **_):
    """Pion and muon slices, each at its own residual range."""
    _errorbar(ax, _within(o["mu_own"], x_max), "s", MUON_COLOR, "Muon", filled=False)
    _errorbar(ax, _within(o["pi"], x_max), "o", SAMPLE_COLORS[o["sample"]], "Pion")


def _step2(ax, o, x_max=None, **_):
    """Muon slices at the pion residual range of equal betagamma, with the pion slices."""
    _errorbar(ax, _within(o["mu"], x_max), "s", MUON_COLOR, r"Muon (equal $\beta\gamma$)",
              filled=False)
    _errorbar(ax, _within(o["pi"], x_max), "o", SAMPLE_COLORS[o["sample"]], "Pion")
    _window_and_join(ax)


def _step3(ax, o, **_):
    """Local fit to the muon slices around the offset window (same model as for the whole
    muon range, in muon residual range), evaluated at the residual range of each pion slice
    (no error bars; step 4 shows the differences with their errors)."""
    lo, hi = PION_OVERLAP
    xlim = (lo - MUON_LOCAL_FIT_MARGIN - 1.0, hi + MUON_LOCAL_FIT_MARGIN + 1.0)
    mu, local = o["mu"], o["mu_local"]
    mz = mu[(mu["rr_center"] >= xlim[0]) & (mu["rr_center"] <= xlim[1])]
    r_mu = np.linspace(local["rr_center"].min(), local["rr_center"].max(), 200)
    x = betagamma_to_rr(211, rr_to_betagamma(13, r_mu))
    f = eval_shift_fit(r_mu, o["popt"], o["model"])
    ax.axvspan(lo, hi, color="gray", alpha=0.15, label="Offset window")
    ax.plot(mz["rr_center"], mz["shift"], "s", color=MUON_COLOR, mfc="none", ms=6,
            label=r"Muon (equal $\beta\gamma$)")
    ax.plot(x, f, color="black", lw=1.5, label=f"Local muon fit ({o['model']})")
    ax.vlines(o["r_pi"], o["mu_fit"], o["pi_shift"], color="black", lw=0.8, alpha=0.5,
              label=r"Pion $-$ muon")
    ax.plot(o["r_pi"], o["mu_fit"], "D", color="black", ms=5,
            label="Muon fit at the pion residual range")
    ax.plot(o["r_pi"], o["pi_shift"], "o", color=SAMPLE_COLORS[o["sample"]], ms=6, label="Pion")
    visible = np.concatenate([o["pi_shift"], mz["shift"].to_numpy(), f])
    pad = 0.08 * (visible.max() - visible.min())
    ax.set_xlim(xlim)
    ax.set_ylim(visible.min() - pad, visible.max() + pad)


def _step4(ax, o, **_):
    """The pion - muon fit differences and their weighted mean (weights 1 / pion error^2),
    which is the offset."""
    lo, hi = PION_OVERLAP
    ax.axhline(0.0, color="gray", ls=":", lw=1)
    ax.errorbar(o["r_pi"], o["diff"], yerr=o["pi_err"], fmt="o", color=SAMPLE_COLORS[o["sample"]],
                label=r"Pion $-$ muon fit at each pion slice", **EB)
    ax.axhline(o["offset"], color="black", lw=1.5, label=r"Weighted mean: offset $\delta$")
    ax.set_xlim(lo - 1.5, hi + 1.5)
    _text(ax, rf"$\delta$ = {o['offset']:+.4f} MeV/cm")


def _step5(ax, o, x_max=None, show_fit=True, **_):
    """Points of the pion shift fit: pion slices up to PION_RR_JOIN and muon slices + offset
    beyond it, with the pion shift fit."""
    color = SAMPLE_COLORS[o["sample"]]
    pts = _within(o["points"], x_max)
    before = _within(o["mu"][o["mu"]["rr_center"] > PION_RR_JOIN], x_max)
    _errorbar(ax, before, "s", MUON_COLOR, "Muon before offset", filled=False, alpha=0.35)
    _errorbar(ax, pts[pts["rr_center"] > PION_RR_JOIN], "s", color, r"Muon + $\delta$",
              filled=False)
    _errorbar(ax, pts[pts["rr_center"] <= PION_RR_JOIN], "o", color, "Pion")
    f = tpc_fits["pion"][o["sample"]]
    popt = f["shift"].get((o["plane"], o["tpc"]), (None, None))[0]
    if show_fit and popt is not None:
        xs = np.linspace(pts["rr_center"].min(), pts["rr_center"].max(), 400)
        ax.plot(xs, eval_shift_fit(xs, popt, f["shift_model"], rr_floor=f["rr_floor"][o["tpc"]]),
                color="black", lw=1.8, label="Pion shift fit")
    _window_and_join(ax)


# Step number -> (title, drawing function, x label, y label)
OFFSET_STEPS = {
    1: ("Step 1: pion and muon slices", _step1, "Residual range [cm]", SHIFT_LABEL),
    2: (r"Step 2: muon slices at the pion residual range of equal $\beta\gamma$", _step2,
        "Pion residual range [cm]", SHIFT_LABEL),
    3: ("Step 3: local muon fit at the pion slices", _step3, "Pion residual range [cm]",
        SHIFT_LABEL),
    4: ("Step 4: offset = weighted mean of the differences", _step4,
        "Pion residual range [cm]", r"Pion $-$ muon shift [MeV/cm]"),
    5: ("Step 5: muon slices + offset, joined to the pion slices", _step5,
        "Pion residual range [cm]", SHIFT_LABEL),
}


def plot_offset_step(sample, step, planes=(0, 1, 2), tpc=PARAM_TPC, x_max=None, show_fit=True,
                     inputs=None):
    """One figure for one step of one sample: a column per plane, with a common legend below.
    x_max limits the residual range of steps 1, 2 and 5. inputs (plane -> offset_inputs) can be
    passed to avoid recomputing them for every step."""
    if inputs is None:
        inputs = {plane: offset_inputs(sample, plane, tpc) for plane in planes}
    title, draw, xlabel, ylabel = OFFSET_STEPS[step]
    fig, axes = plt.subplots(1, len(planes), figsize=(6 * len(planes), 6.2), squeeze=False,
                             layout="constrained")
    axes = axes[0]
    for ax, plane in zip(axes, planes):
        draw(ax, inputs[plane], x_max=x_max, show_fit=show_fit)
        ax.set_title(f"Plane {plane}", fontsize=14)
        ax.set_xlabel(xlabel, fontsize=13)
        ax.grid(True, ls=":", alpha=0.5)
        ax.tick_params(labelsize=11)
    axes[0].set_ylabel(ylabel, fontsize=13)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside lower center", ncol=min(len(labels), 4),
               fontsize=11, frameon=False)
    fig.suptitle(f"{title} ({sample})", fontsize=16, fontweight="bold")
    save_figure(fig, f"convolution_fit_muon_extension_rr_pion_muon_offset_step{step}_{sample}.pdf")
    plt.show()


def run_offset_steps(samples=SAMPLES, steps=tuple(OFFSET_STEPS), planes=(0, 1, 2),
                     tpc=PARAM_TPC, x_max=None, show_fit=True):
    """Step-by-step figure of the pion - muon offset for each sample, and a summary of the
    offsets. Only meaningful for pions with PION_SHIFT_FROM_MUONS."""
    if not (PARTICLE == "pion" and PION_SHIFT_FROM_MUONS):
        print("The muon offset is only used for PARTICLE = 'pion' with PION_SHIFT_FROM_MUONS = True")
        return None
    for sample in samples:
        plot_offset_steps(sample, steps, planes, tpc, x_max, show_fit)
    rows = []
    for sample in samples:
        for plane in planes:
            o = offset_inputs(sample, plane, tpc)
            rows.append(dict(sample=sample, plane=plane, offset=o["offset"]))
    return pd.DataFrame(rows)


def plot_offset_steps(sample, steps=tuple(OFFSET_STEPS), planes=(0, 1, 2), tpc=PARAM_TPC,
                      x_max=None, show_fit=True):
    """One separate figure (and file) per step for one sample. The inputs are computed once and
    shared by all steps."""
    inputs = {plane: offset_inputs(sample, plane, tpc) for plane in planes}
    for step in steps:
        plot_offset_step(sample, step, planes, tpc, x_max, show_fit, inputs=inputs)


offset_summary = run_offset_steps()
# Other options, e.g.:
#   run_offset_steps(steps=(3, 4))                      # only some steps
#   run_offset_steps(x_max=150)                         # limit the residual range of steps 1, 2, 5
offset_summary

In [ ]:
# ======================================================================
# Fits of all planes on one plot, and by plane (MC)
# ======================================================================
def _in_range(df, col, x_limits):
    return df if x_limits is None else df[(df[col] >= x_limits[0]) & (df[col] <= x_limits[1])]


def _zoom_suffix(x_limits):
    return f"_zoom_{x_limits[0]}_{x_limits[1]}".replace(".", "p") if x_limits else ""


def _format_plane(ax, plane, x_limits, y_visible, xlabel, ylabel, legend_loc):
    """Zoom (when x_limits is given), title, labels, legend and grid of one plane panel."""
    if x_limits is not None:
        ax.set_xlim(x_limits)
        if y_visible:
            pad = 0.1 * (max(y_visible) - min(y_visible))
            ax.set_ylim(min(y_visible) - pad, max(y_visible) + pad)
    ax.set_title(f"Plane {plane}", fontsize=18, pad=10)
    ax.set_xlabel(xlabel, fontsize=18)
    if plane == 0:
        ax.set_ylabel(ylabel, fontsize=18)
    if ax.get_legend_handles_labels()[0]:
        ax.legend(fontsize=14, loc=legend_loc, framealpha=0.9)
    ax.grid(True, ls=":", alpha=0.5)
    ax.set_box_aspect(1)


def plot_all_planes(particle, fits, out_prefix=None):
    """sigma_G fits vs theoretical MPV of every (plane, TPC) on one plot:
    color by plane, line style by TPC. fits: tpc_fits[particle]["MC"]."""
    out_prefix = out_prefix or f"convolution_fit_muon_extension_{particle}"
    fig, ax = plt.subplots(figsize=(8, 8))
    for plane in range(3):
        for tpc in (0, 1, -1):
            popt, _ = fits["sigma"].get((plane, tpc), (None, None))
            res = sigma_slices(particle, "MC", plane, tpc)
            if popt is None or not len(res):
                continue
            style = TPC_STYLES[tpc]
            xs = np.linspace(res["mpv_theory"].min(), res["mpv_theory"].max(), 200)
            ax.plot(xs, power_law(xs, *popt), ls=style["linestyle"], color=PLANE_COLORS[plane],
                    lw=2.0, label=f"{style['label_prefix']}, Plane {plane}")
    ax.set_xlabel("Theoretical MPV [MeV/cm]", fontsize=18)
    ax.set_ylabel(SIGMA_LABEL, fontsize=18)
    ax.legend(fontsize=14, loc="upper left", framealpha=0.9)
    ax.grid(True, ls=":", alpha=0.5)
    ax.set_box_aspect(1)
    fig.tight_layout()
    save_figure(fig, f"{out_prefix}_all_planes.pdf")
    plt.show()


def plot_all_planes_shift(particle, fits, out_prefix=None):
    """Shift fits vs residual range of every (plane, TPC) on one plot:
    color by plane, line style by TPC. fits: tpc_fits[particle]["MC"]."""
    out_prefix = out_prefix or f"convolution_fit_muon_extension_{particle}"
    fig, ax = plt.subplots(figsize=(8, 8))
    for plane in range(3):
        for tpc in (0, 1, -1):
            pts = fits["points"].get((plane, tpc))
            popt = fits["shift"].get((plane, tpc), (None, None))[0]
            if popt is None or pts is None or not len(pts):
                continue
            style = TPC_STYLES[tpc]
            xs = np.linspace(pts["rr_center"].min(), pts["rr_center"].max(), 200)
            ys = eval_shift_fit(xs, popt, fits["shift_model"], rr_floor=fits["rr_floor"][tpc])
            ax.plot(xs, ys, ls=style["linestyle"], color=PLANE_COLORS[plane], lw=2.0,
                    label=f"{style['label_prefix']}, Plane {plane}")
    if particle == "pion" and PION_SHIFT_FROM_MUONS:
        ax.axvline(PION_RR_JOIN, color="gray", ls="-.", lw=1, label="End of pion points")
    ax.set_xlabel("Residual range [cm]", fontsize=18)
    ax.set_ylabel(SHIFT_LABEL, fontsize=18)
    ax.legend(fontsize=14, loc="upper right", framealpha=0.9)
    ax.grid(True, ls=":", alpha=0.5)
    ax.set_box_aspect(1)
    fig.tight_layout()
    save_figure(fig, f"{out_prefix}_all_planes_shift.pdf")
    plt.show()


def plot_by_plane(particle, fits, x_limits=None, out_prefix=None):
    """sigma_G vs theoretical MPV, one panel per plane: slices and fits of TPC 0, TPC 1 and
    both TPCs. x_limits zooms on an MPV range [MeV/cm]. fits: tpc_fits[particle]["MC"]."""
    out_prefix = out_prefix or f"convolution_fit_muon_extension_{particle}"
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
    for plane, ax in enumerate(axes):
        y_visible = []
        for tpc in (0, 1, -1):
            res = sigma_slices(particle, "MC", plane, tpc)
            if not len(res):
                continue
            popt, _ = fits["sigma"].get((plane, tpc), (None, None))
            style, color = TPC_STYLES[tpc], TPC_COLORS[tpc]
            pts = _in_range(res, "mpv_theory", x_limits)
            if len(pts):
                ax.errorbar(pts["mpv_theory"], pts["gsigma"], yerr=pts["gsigma_err"],
                            fmt=style["marker"], color=color, ms=4, capsize=2,
                            label=f"{style['label_prefix']} slices")
                y_visible += [(pts["gsigma"] - pts["gsigma_err"]).min(),
                              (pts["gsigma"] + pts["gsigma_err"]).max()]
            if popt is not None:
                lo, hi = x_limits or (res["mpv_theory"].min(), res["mpv_theory"].max())
                xs = np.linspace(lo, hi, 200)
                ys = power_law(xs, *popt)
                ax.plot(xs, ys, ls=style["linestyle"], color=color, lw=1.8,
                        label=f"{style['label_prefix']} fit")
                y_visible += [ys.min(), ys.max()]
        _format_plane(ax, plane, x_limits, y_visible, "Theoretical MPV [MeV/cm]", SIGMA_LABEL,
                      "upper left")
    fig.tight_layout()
    save_figure(fig, f"{out_prefix}_by_plane{_zoom_suffix(x_limits)}.pdf")
    plt.show()


def plot_by_plane_shift(particle, fits, x_limits=None, out_prefix=None):
    """Shift vs residual range, one panel per plane: shift points and fits of TPC 0, TPC 1 and
    both TPCs. For pions with PION_SHIFT_FROM_MUONS, the points beyond PION_RR_JOIN (open
    markers) are muon slices + offset. x_limits zooms on a residual range [cm].
    fits: tpc_fits[particle]["MC"]."""
    out_prefix = out_prefix or f"convolution_fit_muon_extension_{particle}"
    joined = particle == "pion" and PION_SHIFT_FROM_MUONS
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
    for plane, ax in enumerate(axes):
        y_visible = []
        for tpc in (0, 1, -1):
            pts_all = fits["points"].get((plane, tpc))
            if pts_all is None or not len(pts_all):
                continue
            popt = fits["shift"].get((plane, tpc), (None, None))[0]
            style, color = TPC_STYLES[tpc], TPC_COLORS[tpc]
            pts = _in_range(pts_all, "rr_center", x_limits)
            if len(pts):
                from_muon = ((pts["rr_center"] > PION_RR_JOIN).to_numpy() if joined
                             else np.zeros(len(pts), dtype=bool))
                for part, filled, label in ((pts[~from_muon], True, f"{style['label_prefix']} slices"),
                                            (pts[from_muon], False, "_nolegend_")):
                    if len(part):
                        ax.errorbar(part["rr_center"], part["shift"], yerr=part["shift_err"],
                                    fmt=style["marker"], color=color,
                                    mfc=color if filled else "none", ms=4, capsize=2, label=label)
                y_visible += [(pts["shift"] - pts["shift_err"]).min(),
                              (pts["shift"] + pts["shift_err"]).max()]
            if popt is not None:
                lo, hi = x_limits or (pts_all["rr_center"].min(), pts_all["rr_center"].max())
                xs = np.linspace(lo, hi, 200)
                ys = eval_shift_fit(xs, popt, fits["shift_model"], rr_floor=fits["rr_floor"][tpc])
                ax.plot(xs, ys, ls=style["linestyle"], color=color, lw=1.8,
                        label=f"{style['label_prefix']} fit")
                y_visible += [ys.min(), ys.max()]
        if joined:
            ax.axvline(PION_RR_JOIN, color="gray", ls="-.", lw=1, label="End of pion points")
        _format_plane(ax, plane, x_limits, y_visible, "Residual range [cm]", SHIFT_LABEL,
                      "upper right")
    fig.tight_layout()
    save_figure(fig, f"{out_prefix}_shift_by_plane{_zoom_suffix(x_limits)}.pdf")
    plt.show()

In [ ]:
# ======================================================================
# Fits of all planes on one plot, and by plane (MC)
# ======================================================================
def _in_range(df, col, x_limits):
    return df if x_limits is None else df[(df[col] >= x_limits[0]) & (df[col] <= x_limits[1])]


def _zoom_suffix(x_limits):
    return f"_zoom_{x_limits[0]}_{x_limits[1]}".replace(".", "p") if x_limits else ""


def _format_plane(ax, plane, x_limits, y_visible, xlabel, ylabel, legend_loc):
    """Zoom (when x_limits is given), title, labels, legend and grid of one plane panel."""
    if x_limits is not None:
        ax.set_xlim(x_limits)
        if y_visible:
            pad = 0.1 * (max(y_visible) - min(y_visible))
            ax.set_ylim(min(y_visible) - pad, max(y_visible) + pad)
    ax.set_title(f"Plane {plane}", fontsize=18, pad=10)
    ax.set_xlabel(xlabel, fontsize=18)
    if plane == 0:
        ax.set_ylabel(ylabel, fontsize=18)
    if ax.get_legend_handles_labels()[0]:
        ax.legend(fontsize=14, loc=legend_loc, framealpha=0.9)
    ax.grid(True, ls=":", alpha=0.5)
    ax.set_box_aspect(1)


def plot_all_planes(particle, fits, out_prefix=None):
    """sigma_G fits vs theoretical MPV of every (plane, TPC) on one plot:
    color by plane, line style by TPC. fits: tpc_fits[particle]["MC"]."""
    out_prefix = out_prefix or f"convolution_fit_muon_extension_{particle}"
    fig, ax = plt.subplots(figsize=(8, 8))
    for plane in range(3):
        for tpc in (0, 1, -1):
            popt, _ = fits["sigma"].get((plane, tpc), (None, None))
            res = sigma_slices(particle, "MC", plane, tpc)
            if popt is None or not len(res):
                continue
            style = TPC_STYLES[tpc]
            xs = np.linspace(res["mpv_theory"].min(), res["mpv_theory"].max(), 200)
            ax.plot(xs, power_law(xs, *popt), ls=style["linestyle"], color=PLANE_COLORS[plane],
                    lw=2.0, label=f"{style['label_prefix']}, Plane {plane}")
    ax.set_xlabel("Theoretical MPV [MeV/cm]", fontsize=18)
    ax.set_ylabel(SIGMA_LABEL, fontsize=18)
    ax.legend(fontsize=14, loc="upper left", framealpha=0.9)
    ax.grid(True, ls=":", alpha=0.5)
    ax.set_box_aspect(1)
    fig.tight_layout()
    save_figure(fig, f"{out_prefix}_all_planes.pdf")
    plt.show()


def plot_all_planes_shift(particle, fits, out_prefix=None):
    """Shift fits vs residual range of every (plane, TPC) on one plot:
    color by plane, line style by TPC. fits: tpc_fits[particle]["MC"]."""
    out_prefix = out_prefix or f"convolution_fit_muon_extension_{particle}"
    fig, ax = plt.subplots(figsize=(8, 8))
    for plane in range(3):
        for tpc in (0, 1, -1):
            pts = fits["points"].get((plane, tpc))
            popt = fits["shift"].get((plane, tpc), (None, None))[0]
            if popt is None or pts is None or not len(pts):
                continue
            style = TPC_STYLES[tpc]
            xs = np.linspace(pts["rr_center"].min(), pts["rr_center"].max(), 200)
            ys = eval_shift_fit(xs, popt, fits["shift_model"], rr_floor=fits["rr_floor"][tpc])
            ax.plot(xs, ys, ls=style["linestyle"], color=PLANE_COLORS[plane], lw=2.0,
                    label=f"{style['label_prefix']}, Plane {plane}")
    if particle == "pion" and PION_SHIFT_FROM_MUONS:
        ax.axvline(PION_RR_JOIN, color="gray", ls="-.", lw=1, label="End of pion points")
    ax.set_xlabel("Residual range [cm]", fontsize=18)
    ax.set_ylabel(SHIFT_LABEL, fontsize=18)
    ax.legend(fontsize=14, loc="lower right", framealpha=0.9)
    ax.grid(True, ls=":", alpha=0.5)
    ax.set_box_aspect(1)
    fig.tight_layout()
    save_figure(fig, f"{out_prefix}_all_planes_shift.pdf")
    plt.show()


def plot_by_plane(particle, fits, x_limits=None, out_prefix=None):
    """sigma_G vs theoretical MPV, one panel per plane: slices and fits of TPC 0, TPC 1 and
    both TPCs. x_limits zooms on an MPV range [MeV/cm]. fits: tpc_fits[particle]["MC"]."""
    out_prefix = out_prefix or f"convolution_fit_muon_extension_{particle}"
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
    for plane, ax in enumerate(axes):
        y_visible = []
        for tpc in (0, 1, -1):
            res = sigma_slices(particle, "MC", plane, tpc)
            if not len(res):
                continue
            popt, _ = fits["sigma"].get((plane, tpc), (None, None))
            style, color = TPC_STYLES[tpc], TPC_COLORS[tpc]
            pts = _in_range(res, "mpv_theory", x_limits)
            if len(pts):
                ax.errorbar(pts["mpv_theory"], pts["gsigma"], yerr=pts["gsigma_err"],
                            fmt=style["marker"], color=color, ms=4, capsize=2,
                            label=f"{style['label_prefix']} slices")
                y_visible += [(pts["gsigma"] - pts["gsigma_err"]).min(),
                              (pts["gsigma"] + pts["gsigma_err"]).max()]
            if popt is not None:
                lo, hi = x_limits or (res["mpv_theory"].min(), res["mpv_theory"].max())
                xs = np.linspace(lo, hi, 200)
                ys = power_law(xs, *popt)
                ax.plot(xs, ys, ls=style["linestyle"], color=color, lw=1.8,
                        label=f"{style['label_prefix']} fit")
                y_visible += [ys.min(), ys.max()]
        _format_plane(ax, plane, x_limits, y_visible, "Theoretical MPV [MeV/cm]", SIGMA_LABEL,
                      "upper left")
    fig.tight_layout()
    save_figure(fig, f"{out_prefix}_by_plane{_zoom_suffix(x_limits)}.pdf")
    plt.show()


def plot_by_plane_shift(particle, fits, x_limits=None, out_prefix=None):
    """Shift vs residual range, one panel per plane: shift points and fits of TPC 0, TPC 1 and
    both TPCs. For pions with PION_SHIFT_FROM_MUONS, the points beyond PION_RR_JOIN (open
    markers) are muon slices + offset. x_limits zooms on a residual range [cm].
    fits: tpc_fits[particle]["MC"]."""
    out_prefix = out_prefix or f"convolution_fit_{particle}"
    joined = particle == "pion" and PION_SHIFT_FROM_MUONS
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
    for plane, ax in enumerate(axes):
        y_visible = []
        for tpc in (0, 1, -1):
            pts_all = fits["points"].get((plane, tpc))
            if pts_all is None or not len(pts_all):
                continue
            popt = fits["shift"].get((plane, tpc), (None, None))[0]
            style, color = TPC_STYLES[tpc], TPC_COLORS[tpc]
            pts = _in_range(pts_all, "rr_center", x_limits)
            if len(pts):
                from_muon = ((pts["rr_center"] > PION_RR_JOIN).to_numpy() if joined
                             else np.zeros(len(pts), dtype=bool))
                for part, filled, label in ((pts[~from_muon], True, f"{style['label_prefix']} slices"),
                                            (pts[from_muon], False, "_nolegend_")):
                    if len(part):
                        ax.errorbar(part["rr_center"], part["shift"], yerr=part["shift_err"],
                                    fmt=style["marker"], color=color,
                                    mfc=color if filled else "none", ms=4, capsize=2, label=label)
                y_visible += [(pts["shift"] - pts["shift_err"]).min(),
                              (pts["shift"] + pts["shift_err"]).max()]
            if popt is not None:
                lo, hi = x_limits or (pts_all["rr_center"].min(), pts_all["rr_center"].max())
                xs = np.linspace(lo, hi, 200)
                ys = eval_shift_fit(xs, popt, fits["shift_model"], rr_floor=fits["rr_floor"][tpc])
                ax.plot(xs, ys, ls=style["linestyle"], color=color, lw=1.8,
                        label=f"{style['label_prefix']} fit")
                y_visible += [ys.min(), ys.max()]
        if joined:
            ax.axvline(PION_RR_JOIN, color="gray", ls="-.", lw=1, label="End of pion points")
        _format_plane(ax, plane, x_limits, y_visible, "Residual range [cm]", SHIFT_LABEL,
                      "lower right")
    fig.tight_layout()
    save_figure(fig, f"{out_prefix}_shift_by_plane{_zoom_suffix(x_limits)}.pdf")
    plt.show()

In [ ]:
for particle in PLOT_PARTICLES:
    plot_all_planes(particle, tpc_fits[particle]["MC"])
    plot_all_planes_shift(particle, tpc_fits[particle]["MC"])

In [ ]:
for particle in PLOT_PARTICLES:
    plot_by_plane(particle, tpc_fits[particle]["MC"])
    plot_by_plane_shift(particle, tpc_fits[particle]["MC"])
# Zoom, e.g.: plot_by_plane_shift("pion", tpc_fits["pion"]["MC"], x_limits=(2, 40))